In [1]:
import pandas as pd, numpy as np, torch, torchvision, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import sys
sys.path.insert(0, '../')
from util import *
from models.cnn import CNN

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)


# access each scene folder
scenarios = [f"../CSV_Files/scene{i+1}/" for i in range(6)]



device: NVIDIA A30


# Helper functions

In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    """Load one scenario, run preprocessing, return tensors + shapes."""
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(device, seed=0):
    """Fresh CNN with Xavier init, to match MATLAB's Glorot default."""
    torch.manual_seed(seed)
    model = CNN(n_classes=8).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    """MATLAB-style 'shuffle once' then fixed-order iteration."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def run_scenario(scenario_idx, scenario_dir, model_cls, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    # ---- compute metrics setup ----
    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # ---- TRAINING with timer ----
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- INFERENCE timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- TEST accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        # ---- new compute fields ----
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }




In [3]:
results = []
for i, sc_dir in enumerate(scenarios, start=1):
    r = run_scenario(i, sc_dir, model_cls=CNN, device=device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (CNN) ===
  params: 644  breakdown: {'conv': 36, 'bn': 24, 'fc': 584}
  ep   1 | train loss 1.6700 acc 0.413 | val loss 1.4308 acc 0.514
  ep  10 | train loss 0.8030 acc 0.690 | val loss 0.8046 acc 0.691
  ep  20 | train loss 0.6797 acc 0.735 | val loss 0.6879 acc 0.756
  ep  30 | train loss 0.5756 acc 0.785 | val loss 0.5903 acc 0.814
  ep  40 | train loss 0.5400 acc 0.796 | val loss 0.5552 acc 0.833
  ep  50 | train loss 0.4915 acc 0.820 | val loss 0.5203 acc 0.858
  ep  60 | train loss 0.4764 acc 0.826 | val loss 0.5056 acc 0.863
  ep  70 | train loss 0.4543 acc 0.838 | val loss 0.5015 acc 0.869
  TEST: acc=0.8644  P=0.8706  R=0.8644  F1=0.8631
  COMPUTE: train=48.8s  inf=0.000ms/sample  peak_mem=17.4MB

=== Scenario 2 (CNN) ===
  params: 644  breakdown: {'conv': 36, 'bn': 24, 'fc': 584}
  ep   1 | train loss 1.8310 acc 0.353 | val loss 1.6505 acc 0.466
  ep  10 | train loss 0.9526 acc 0.652 | val loss 0.9589 acc 0.679
  ep  20 | train loss 0.7906 acc 0.710 | val los

In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv("cnn_results_by_scenario.csv", index=False)


 scenario model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1   CNN     7858    0.8644     0.8706  0.8644 0.8631       644      48.77             0.0002         17.4
        2   CNN     3939    0.8275     0.8381  0.8275 0.8272       644      24.88             0.0002         17.4
        3   CNN      786    0.6844     0.7139  0.6844 0.6844       644       7.53             0.0002         17.4
        4   CNN      391    0.6019     0.6294  0.6019 0.5991       644       5.32             0.0003         17.4
        5   CNN      238    0.5350     0.5856  0.5350 0.5449       644       4.44             0.0005         17.4
        6   CNN       77    0.3700     0.3964  0.3700 0.3774       644       3.89             0.0002         17.4


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  (n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (n_train=7858, acc=0.864)
[[149   6   0  38   0   0   6   1]
 [  0 186   0   5   0   1   0   8]
 [  0   0 195   0   0   1   0   4]
 [ 21   5   2 150   0   0  22   0]
 [  0   0   0   0 200   0   0   0]
 [  0   7   5   0   0 138   0  50]
 [  1   0   0  11   1   0 187   0]
 [  0  13   2   1   0   6   0 178]]

Scenario 2  (n_train=3939, acc=0.828)
[[147   4   0  39   0   2   7   1]
 [  2 147   1   6   0   6   1  37]
 [  0   0 195   0   0   2   0   3]
 [ 46   5   0 133   0   0  16   0]
 [  0   0   0   0 200   0   0   0]
 [  1   9   5   0   0 136   0  49]
 [  1   0   0  20   0   0 179   0]
 [  3   4   3   1   0   2   0 187]]

Scenario 3  (n_train=786, acc=0.684)
[[107   1   0  52   0   2  22  16]
 [ 26  99   5   6   0   9   0  55]
 [  1   0 172   0   1  10   1  15]
 [ 58   1   1 106   0   1  24   9]
 [  0   0   0   0 199   1   0   0]
 [  2   7  20   2   2 104   0  63]
 [  9   0   0  45   0   0 146   0]
 [  8   7  18   2   1   2   0 162]]

Scenario 4  (n_train=391, acc=0.602)
[[ 